# Counting ships in the port of Constanța with Sentinel-1
*Luca Tarchila, Paulian Oprescu, Phuong Nguyen, Ravian Hendriks · Geoscripting, WUR, 2026*

Constanța is the largest port in the Black Sea. After Russia invaded Ukraine in 2022, it became an important logistic hub for the grain of Ukraine. Did that change how many vessels there were in the port?

We count vessels in the port and the offshore anchorage on Sentinel-1 radar images from 2019 to 2025, uusing Google Earth Engine. Radar can see strough clouds and works at night. Ships show up as bright spots on dark water.

**Questions**
1. How many vessels can we detect with a simple brightness threshold?
2. Did the number of vessels change after the war started?

**Table of contents**
* **Settings:** all variables that are used in one cell
* **Step 1-5:** water mask, images, detection, time series, before vs after
* **What we tried:** methods that didn't work and why
* Results are saved in `data/`, so rerunning the notebook skips the slow calculation part.

## Pixi install instructions

## Before you start: Google Earth Engine access
https://geoscripting-wur.github.io/Python-GEE/
This project uses Google Earth Engine. To work with Google Earth Engine (GEE), you will need a Google account for authorization. The authentication process can be initiated by calling the Authenticate function from the ee module. Follow these steps:

1. After running the Authenticate function below, a URL will be printed in the output.
2. Click on the URL, which will redirect you to a Google account login page.
3. Log in to your Google account to create an authorization token.
4. When prompted with a warning stating that Google has not verified the app, click “Continue”. This warning is shown because the code we are about to run is not verified by Google, ensuring your account’s security.
5. On the authorization page, select all the checkboxes. This grants permission to access GEE data and manage data and permissions in the cloud storage where the analysis will be performed.
6. Finally, you will see an authorization code. Copy the code and paste it into the text box that appears after running the Authenticate function. Press Enter to complete the authentication process.

In [ ]:
import ee
ee.Authenticate()

Now that you have authenticated your account, run the next cell which will initialize

In [ ]:
ee.Initialize()

Project setup

In [ ]:
# Your own GEE project ID (see 'Before you start' above)
gee_project = None

## Settings
All choices for the analysis are in this cell. Change a value here and run the notebook again. Nothing further down needs editing.

In [ ]:
# Google Earth Engine project (change to your own)
gee_project = 'tutorial-geoscripting-paulian'


# Study area
# Bounding box [west, south, east, north] in degrees (WGS84)
# Covers Constanța port and the offshore anchorage
bbox = [28.64, 44.08, 28.88, 44.18]
# Longitude that split the box: port to the west, anchorage to the east
split_lon = 28.70

# Time period
start_date = '2019-01-01' # first day included
end_date   = '2026-01-01' # not included

# Sentinel-1
# Relatiive orbit 58 covers the area most often. Using one orbit
# keeps the viewing angle the same in every image.
orbit = 58

# Ship detection
threshold = -15        # db: water pixels brighter than this are ship pixels
min_pixels = 8         # smallest group of bright pixels that counts as a ship 
water_buffer = 10      # m: shrink the water mask inward to 

# Methods we tried (see 'What we tried')
k = 4                  # adaptive and CFAR: standard deviations above the background
guard = 20             # CFAR: radius around the pixel that is ignored
outer = 25             # Cfar: outer radius of the background ring, in pixels

# Files and folders
water_shp = 'masks/clip_water.shp'
data_dir = 'data'
output_dir = 'output'

## 2. Setup
Load the packages, connect to Google Earth Enginge and create the folders for results.

In [ ]:
import os
import geemap
from functions.detection import load_water, load_s1, detect_ships, ship_counts, sea_threshold
import pandas as pd
import matplotlib.pyplot as plt

# Log in to Google Earth Engine (only need to do it once per environment)
# ee.Authenticate()

# Connect to Google Earth Engine
ee.Initialize(project=gee_project)

# Create the data and output folders if they don't exist yet
os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# 3. Water mask
In radar images, ships light up amongst bodies of water, but so does land, buildings and docks. To only count ships we will look at water pixels only.

We use the OpenStreetMap water polygons ([osmdata.openstreetmap.de](https://osmdata.openstreetmap.de/data/water-polygons.html)), clipped to our area of interest.

The mask shrinks a few metres inward, so the bright edges of land and docks are not mistaken for ships.

In [ ]:
# https://geemap.org/cheatsheet/#add-earth-engine-layers
# https://geoscripting-wur.github.io/Python-GEE/


water = load_water(water_shp, bbox, buffer=10)
port = ee.Geometry.Rectangle([bbox[0], bbox[1], split_lon, bbox[3]])
anchorage = ee.Geometry.Rectangle([split_lon, bbox[1], bbox[2], bbox[3]])
m = geemap.Map()
m.centerObject(water, 12)
m.addLayer(water, {'color': 'blue'}, ' water mask')
m

## 4. Sentinel-1 images
Sentinel-1 is a radar satellite. The satellite records two polarisations:

- **VV**:
- **VH**

**The sea.**

**Ships.**

. VH and VV
* What polarisation is
* How the sea behaves
* How ships behave
* Why that matters
* Trade off

We use Sentinel-1 radar images with the VH band. VH gives more contrast between ships and water than VV.


In [ ]:
# https://geemap.org/notebooks/04_split_panel_map/
aoi = ee.Geometry.Rectangle(bbox)
s1 = load_s1(aoi, start_date, end_date, orbit)
print('Number of Sentinel-1 scenes:', s1.size().getInfo())

s1_full = s1.filter(ee.Filter.contains('.geo', aoi))
image_vv_vh = s1_full.first().clip(aoi)

left_layer = geemap.ee_tile_layer(image_vv_vh.select('VH'), {"min":-25, "max": 0})
right_layer = geemap.ee_tile_layer(image_vv_vh.select('VV'), {"min":-25, "max": 0})

Map = geemap.Map()
Map.centerObject(aoi, 12)
Map.split_map(left_layer, right_layer)
Map


In [ ]:
m = geemap.Map()
m.centerObject(aoi, 12)
m.addLayer(image_vv_vh.select('VV'), {"min": -25, "max": 0}, 'VV')
m

## 5. Ship detection on one scene
To recognize ships in an image, we are testing two methods to find them.
1. **Fixed threshold:** every water pixel brighter than 'threshold' is a ship pixel
2. **CFAR (Constant False Alarm Rate):** a pixel is a ship if it is much brighter than the water around it

Touching bright pixels will form one object, and objects smaller than min_pixels are seen as noise and will be removed.

In [ ]:
image = s1.first()
print('Date', image.date().format('YYYY-MM-dd HH:mm').getInfo())
ships_threshold = detect_ships(image, water, method='threshold', threshold=threshold, min_pixels=min_pixels)
# ships_cfar = detect_ships(image, water, method='cfar', k=2, min_pixels=min_pixels)

print('Ships threshold:', ships_threshold.size().getInfo())
#print('Ships CFAR:', ships_cfar.size().getInfo())

In [ ]:
ships = detect_ships(image, water, method='threshold', threshold=threshold, min_pixels=min_pixels)
ships_port = ships.filterBounds(port)
ships_anchorage = ships.filterBounds(anchorage)
print('Port:', ships_port.size().getInfo(), '| Anchorage:', ships_anchorage.size().getInfo())

map = geemap.Map()
map.centerObject(water, 12)
map.addLayer(water, {'color': 'blue'}, 'water mask')
map.addLayer(ships_port, {'color': 'red'}, 'ships - port')
#map.addLayer(ships_cfar, {'color': 'red'}, ' water mask')
map

## 6. Time series
We count the ships in every scence from 2019 to 2025.
This process will take a few minutes, so the result is saved to 'data/ship_counts.csv. 
The next time you run the notebook, the counts are loaded from that file instead.

In [ ]:
# counts_cfar = ship_counts(s1, water, method='cfar', guard=guard, outer=outer, k=k, min_pixels=min_pixels)
counts_port = ship_counts(s1, water.intersection(port, 1), threshold=threshold, k=k, min_pixels=min_pixels)
counts_anchorage = ship_counts(s1, water.intersection(anchorage, 1), threshold=threshold, k=k, min_pixels=min_pixels)
counts = pd.concat({'port': counts_port['n_ships'], 'anchorage': counts_anchorage['n_ships']}, axis=1)
counts['total'] = counts['port'] + counts['anchorage']

## 7. Before and after the war
Monthly average number of detected ships for both methods. 

In [ ]:
# Monthly average, so the plot is less noisy than 322 single dates
monthly = counts.resample('MS').mean()
war_start = '2022-02-24'

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(monthly.index, monthly['anchorage'], label='Anchorage')


ax.axvline(pd.Timestamp(war_start), color='black', linestyle='--', label='Start of war')
ax.axvline(pd.Timestamp('2023-07-17'), color='red', linestyle=':', label='Russia leaves grain deal')
ax.set_ylabel('Detected ships (monthly mean)')
ax.legend()
fig.savefig(os.path.join(output_dir, 'ship_counts.png'), dpi=150, bbox_inches='tight')
plt.show()

# Average number of ships per image, before and after the start of the war
before = counts[counts.index < war_start].mean()
after = counts[counts.index >= war_start].mean()
print(pd.DataFrame({'before war': before, 'after war': after}).round(1))

counts.to_csv(os.path.join(data_dir, 'ship_counts.csv'))

In [ ]:
import requests
import pandas as pd

# https://www.w3schools.com/PYTHON/ref_requests_get.asp
# https://developers.arcgis.com/rest/services-reference/enterprise/query-feature-service-layer/
# https://docs.python-requests.org/en/latest/user/quickstart/#json-response-content
url = "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/Daily_Ports_Data/FeatureServer/0/query"

# ArcGIS API has a limit of 1000 lines per get, so we reset the request after 1000 gets.
offset = 0
pagesize = 1000

# First count the amount of times port260 is in the database
count = requests.get(url, params={"where": "portid='port260'", "returnCountOnly": "true", "f": "json"})
total = count.json()["count"]

rows = []

while offset < total:
    params = {
        "where": "portid='port260'",
        "outFields": "year,month,day,portcalls",
        "orderByFields": "date",
        "resultOffset": offset,
        "resultRecordCount": pagesize,
        "f": "json",
    }

    data = requests.get(url, params=params).json()

    for f in data["features"]:
        rows.append(f["attributes"])
    offset += pagesize
df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df[["year", "month", "day"]])

csv = df.to_csv("data/portwatch_constanta.csv", index=False)

In [ ]:
print(csv)

In [ ]:
pw = pd.read_csv("data/portwatch_constanta.csv", parse_dates=["date"])

# Basic info
print("Days:", len(pw))
print("From", pw["date"].min().date(), "to", pw["date"].max().date())
print("Missing values:", pw["portcalls"].isna().sum())

# Port calls per day: mean, min, max, etc.
print(pw["portcalls"].describe())

# Total port calls per year
print(pw.groupby("year")["portcalls"].sum())

# Before vs after the start of the war (24 Feb 2022)
before = pw[pw["date"] < "2022-02-24"]["portcalls"].mean()
after = pw[pw["date"] >= "2022-02-24"]["portcalls"].mean()
print(f"Average port calls per day before the war: {before:.1f}")
print(f"Average port calls per day after the war:  {after:.1f}")

In [ ]:
import matplotlib.pyplot as plt

pw = pd.read_csv("data/portwatch_constanta.csv", parse_dates=["date"])
pw = pw[pw["date"] < "2026-01-01"]   # keep 2019-2025, like your Sentinel-1 data

# Add up all days per month
monthly = pw.resample("MS", on="date")["portcalls"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, marker="o", markersize=3)
ax.axvline(pd.Timestamp("2022-02-24"), color="red", linestyle="--",
           label="Start of war (24 Feb 2022)")
ax.set_xlabel("Month")
ax.set_ylabel("Port calls per month")
ax.set_title("Port calls in Constanța (IMF PortWatch)")
ax.legend()
plt.show()

In [ ]:
# https://www.geeksforgeeks.org/python/how-to-filter-dataframe-rows-based-on-the-date-in-pandas/
# https://stackoverflow.com/questions/65471540/get-monthly-average-in-pandas
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html
portwatch = pd.read_csv(data_dir + "/portwatch_constanta.csv")
ships = pd.read_csv(data_dir + "/ship_counts.csv")

portwatch['date'] = pd.to_datetime(portwatch['date'])
ships['date'] = pd.to_datetime(ships['date'])
correct_dates_portwatch = portwatch.loc[portwatch['date'] < end_date]

pw_months_mean = correct_dates_portwatch.resample('MS', on='date')['portcalls'].mean()
total_predict_mean = ships.resample('MS', on='date')['total'].mean()
port_predict_mean = ships.resample('MS', on='date')['port'].mean()
anchorage_predict_mean = ships.resample('MS', on='date')['anchorage'].mean()

# axis=1 puts the four series side by side as columns, one row per month
monthly = pd.concat({'portcalls': pw_months_mean, 'total': total_predict_mean,
                     'port': port_predict_mean, 'anchorage': anchorage_predict_mean}, axis=1)

print(monthly.head())
print(monthly.corr(method='spearman'))

In [ ]:
ships = pd.read_csv("data/ship_counts.csv")
print(ships.columns)

## 8. Timelapse showing vessel traffic change

Showcasing the change in vessels counted after the Russian Federation left the grain deal in 2023.

In [ ]:
# Timelapse of the detected ships around Russia leaving the grain deal (17 July 2023)
# https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.HTML
# https://docs.python.org/3/library/base64.html#base64.b64encode
# https://developer.mozilla.org/en-US/docs/Web/URI/Reference/Schemes/data
import base64
from IPython.display import HTML, display
from functions.detection import make_timelapse

# Specify where to put gif file
gif_path = os.path.join(output_dir, 'timelapse_2023.gif')

# Running timelapse function for capturing the grain deal leave event
make_timelapse(s1.filterDate('2023-04-01', '2023-11-01'), water, aoi, gif_path,
               method='threshold', threshold=threshold, min_pixels=min_pixels)

# Show the GIF in the notebook (embedded as text, the way Positron displays it)
with open(gif_path, 'rb') as f:
    display(HTML(f'<img src="data:image/gif;base64,{base64.b64encode(f.read()).decode()}">'))